# Cloud Provider Analytics: pipeline completo en Google Colab

Corre los cinco steps del pipeline (`bronze_batch`, `bronze_stream`, `silver`, `gold`, `serving`) en Colab, sin Docker, y publica Gold en AstraDB. Después muestra los conteos de `_manifests/` y corre las consultas 1 y 2 contra AstraDB, con la org y las fechas calculadas desde Gold. Usa el dataset del repo (`data/datalake/landing`), así que no hace falta montar Drive.

**Credenciales:** se leen de **Colab Secrets** y nunca se escriben en el notebook ni en Drive. Cargar estos secrets (ícono de la llave, a la izquierda) y habilitarles el acceso a este notebook:

| Secret | Valor |
|---|---|
| `ASTRA_DB_APPLICATION_TOKEN` | Token de aplicación de AstraDB |
| `ASTRA_DB_API_ENDPOINT` | `https://<db_id>-<region>.apps.astra.datastax.com` |

El secure connect bundle se descarga solo con esos dos valores y queda en el disco de la VM (`secrets/` dentro del clon), que se borra al cerrar la sesión. Si en vez del endpoint se prefiere subir un bundle ya descargado, ver la celda de configuración.

El keyspace `cloud_analytics` tiene que existir en Astra antes de correr: se crea desde la consola ([cómo](https://github.com/VieraFederico/tpe-big-data/blob/main/docs/serving.md#crear-el-keyspace-en-astradb)).

Para correrlo: `Runtime → Run all`. Si un step falla, su celda corta la ejecución y el detalle queda en `evidence/logs/`.

## 1. Código y dependencias

Clona el repo e instala las versiones fijas de `config/requirements.txt`, las mismas que usa la imagen de Docker. Spark necesita Java 17.

In [ ]:
import os, sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/VieraFederico/tpe-big-data.git"
BRANCH = "main"
FORCE_CLONE = False
LOCAL_CLONE_DIR = Path.home() / "tpe-big-data"

IN_COLAB = "google.colab" in sys.modules
START_DIR = globals().get("START_DIR", Path.cwd().resolve())

if IN_COLAB or FORCE_CLONE:
    target = Path("/content/tpe-big-data") if IN_COLAB else LOCAL_CLONE_DIR.resolve()
    if not IN_COLAB and START_DIR.is_relative_to(target):
        raise RuntimeError(f"El notebook está dentro de {target}; elegí otro LOCAL_CLONE_DIR.")
    os.chdir(target.parent)
    shutil.rmtree(target, ignore_errors=True)
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} "{target}"
    os.chdir(target)
else:
    root = START_DIR
    while not (root / "config/requirements.txt").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

print("Trabajando en:", Path.cwd())

if IN_COLAB:
    %pip uninstall -y -q dataproc-spark-connect
%pip install -q --only-binary=:all: cassandra-driver
reqs = [line.strip() for line in Path("config/requirements.txt").read_text().splitlines()
        if line.strip() and not line.strip().startswith(("#", "cassandra-driver"))]
%pip install -q {" ".join(reqs)}

!python -c "import cassandra, pyspark; print('cassandra-driver', cassandra.__version__, '| pyspark', pyspark.__version__)"

In [ ]:
import shutil
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!java -version 2>&1 | head -1

## 2. Configuración

Las variables son las de `config/.env.example` ([config/README.md](https://github.com/VieraFederico/tpe-big-data/blob/main/config/README.md)). En Colab no hay `.env`: se definen acá y los secretos salen de Colab Secrets.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

os.environ["SERVING_TARGET"] = "astra"
os.environ["CASSANDRA_KEYSPACE"] = "cloud_analytics"
os.environ["DATALAKE_PATH"] = str(Path.cwd() / "data" / "datalake")
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = userdata.get("ASTRA_DB_APPLICATION_TOKEN")

# Con un bundle subido a mano (desde el panel de archivos), poner acá su ruta;
# así no hace falta el secret del endpoint.
BUNDLE_SUBIDO = ""
if BUNDLE_SUBIDO:
    os.environ["ASTRA_DB_SECURE_BUNDLE_PATH"] = BUNDLE_SUBIDO
else:
    os.environ["ASTRA_DB_API_ENDPOINT"] = userdata.get("ASTRA_DB_API_ENDPOINT")

print("SERVING_TARGET =", os.environ["SERVING_TARGET"])
print("DATALAKE_PATH  =", os.environ["DATALAKE_PATH"])
print("Token cargado  =", bool(os.environ["ASTRA_DB_APPLICATION_TOKEN"]))

In [ ]:
!python -m src.serving.check_connection
assert _exit_code == 0, "No se pudo conectar a AstraDB: revisar los secrets y que exista el keyspace"

## 3. Pipeline

Un step por celda, en el orden de `./infra/run.sh run`. Cada uno deja su log en `evidence/logs/pipeline_<run_id>.log`.

In [ ]:
!python -m src.pipeline --list

### `bronze_batch`: los siete CSV de maestros a Bronze

In [ ]:
!python -m src.pipeline --steps bronze_batch
assert _exit_code == 0, "Falló bronze_batch"

### `bronze_stream`: eventos JSONL con Structured Streaming (watermark, dedupe y checkpoint)

In [ ]:
!python -m src.pipeline --steps bronze_stream
assert _exit_code == 0, "Falló bronze_stream"

### `silver`: reglas de calidad, quarantine y enriquecimiento

In [ ]:
!python -m src.pipeline --steps silver
assert _exit_code == 0, "Falló silver"

### `gold`: gate de reconciliation y los tres marts

In [ ]:
!python -m src.pipeline --steps gold
assert _exit_code == 0, "Falló gold: el detalle del gate está en evidence/quality/<run_id>/reconciliation.csv"

### `serving`: carga de los marts en AstraDB

In [ ]:
!python -m src.pipeline --steps serving
assert _exit_code == 0, "Falló serving"

## 4. Conteos de `_manifests/`

Lo que escribió cada step según su manifest: rows leídas, escritas, en quarantine y duplicadas por dataset, y rows por mart de Gold. Es lo mismo que deja `conteos.csv` en la evidencia de la entrega.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.config import load_settings
from src.evidence.collect import COUNTS_HEADER, manifest_counts

conteos = pd.DataFrame(manifest_counts(load_settings()), columns=COUNTS_HEADER)
conteos.pivot_table(index=["paso", "dataset"], columns="metrica", values="valor", aggfunc="first")

## 5. Consultas 1 y 2 en AstraDB

Igual que `./infra/run.sh queries`: la org es la de mayor costo de 14 días al 2025-08-31 (`src/serving/evidence_params.py`) y el rango son esos 14 días. Imprime la consulta 1 (uso y costo diario por service) y la 2 (top 5 de services por costo de 14 días), y las guarda en `evidence/local/`.

In [ ]:
!python -m src.evidence.queries
assert _exit_code == 0, "Fallaron las consultas"

## 6. Evidencia

Los logs quedan en `evidence/logs/` y las consultas en `evidence/local/`. La última celda los baja en un zip.

In [ ]:
!ls -lh evidence/logs/ evidence/local/
!du -sh data/datalake/*/ 2>/dev/null

In [ ]:
from google.colab import files

!cd evidence && zip -qr /content/evidencia_colab.zip logs local
files.download("/content/evidencia_colab.zip")